# 02: IQPEのビット数と測定精度の相関
### (Correlation between the number of bits in QIPE and measurement accuracy)

---

## 1. Thesis: 科学的問いと仮説
量子位相推定においては、位相を高精度で推定することが重要になります。標準的な量子位相推定（QPE）は複数量子ビットを必要としますが、位相のみが関心対象であり固有状態の再利用が可能な場合、より少ない量子ビットで同等の情報を得られる可能性があります。

**問い**: 補助量子ビットを1量子ビットに制限した場合でも、位相情報を測定確率として逐次的に読み出し、高精度で位相を推定することは可能なのか？また、その推定精度は反復回数にどのように依存するのか？

**仮説**: 反復量子位相推定（Iterative Quantum Phase Estimation, IPE）の標準的実装であるLSB優先方式は、位相 $θ$ を二進展開 $θ=2π(0.b_1​b_2​⋯b_n​)$ で表現した際に、最下位ビット $b_n$ から順番に決定していく方式であり、この手法では補助量子ビット1つのみを用いて $n$ ビットの位相情報を抽出でき、推定誤差はおおよそ使用ビット数 $n$ に対しておおよそ $2π / 2^n$ に比例して減少することが期待される。

## 2. Theoretical Background (理論的背景)

### 2.1 H（Hadamard）ゲート

H ゲートは、量子計算において最も重要なゲートの一つであり、**基底状態と重ね合わせ状態を相互に変換する操作**を担います。量子センシングにおいては、「センサの初期化（準備）」と「位相の読み出し（干渉）」という二つの重要な役割を果たします。

#### 数学的定義

## 3. Implementation (実装)
外部ライブラリ "Qiskit" から `AerSimulator`、`QuantumCircuit` 、`transpile` をインポートします。

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np

def iterative_phase_estimation(true_theta: float, num_bits: int = 6) -> float:
    """反復量子位相推定(IQPE)アルゴリズムを用いて位相を推定します.

    量子位相推定の省リソース版.
    1量子ビットのみを使用して位相を最下位ビットから順に決定していきます.

    Args:
        true_theta (float): 推定対象の真の位相[rad] (0 ~ 2π)
        num_bits (int, optional): 推定精度(ビット数). Defaults to 6.
            精度 ≈ 2π / 2^num_bits (例: 6ビット → 約0.098 rad)

    Returns:
        float: 推定された位相[rad]

    Algorithm:
        各ビットiについて(最下位ビットから):
        1. |+⟩状態を準備
        2. true_theta × 2^i の位相回転を適用
        3. 既に決定したビットによる補正(-estimated_phase × 2^i)
        4. 測定により現在のビット値を決定
        5. 結果に応じて推定位相を更新

    Example:
        >>> # π/4 (約0.785 rad)を推定
        >>> iterative_phase_estimation(np.pi/4, num_bits=6)
        --- IPE algorithm (Bits: 6) ---
            修正した結果: 0.7854
    """
    print(f"\n--- IPE algorithm (Bits: {num_bits}) ---")

    estimated_phase = 0.0
    for i in reversed(range(num_bits)):
        scaling = 2**i
        qc = QuantumCircuit(1, 1)
        qc.h(0)  # H（重ね合わせ）
        qc.rz(true_theta * scaling, 0)  # RZ（位相回転）
        qc.rz(-estimated_phase * scaling, 0)  # RZ (既知位相の打ち消し)
        qc.h(0)  # H（干渉）
        qc.measure(0, 0)  # 測定

        # --- 実行 ---
        sim = AerSimulator()
        simulation_result = sim.run(transpile(qc, sim), shots=1).result()
        measured_bit = int(list(simulation_result.get_counts().keys())[0])

        # 位相の更新
        if measured_bit == 1:
            estimated_phase += (1 / 2 ** (i + 1)) * (2 * np.pi)

    print(f" 修正した結果: {estimated_phase:.4f}")
    return estimated_phase

## 4. Visualization & Analysis (可視化と解析)

## 5. Conclusion & Future Work (結論と展望)